In [1]:
import os
from ift6758.data import load_season, get_game
from ift6758.data.nhl import GAME_TYPE_CODE, get_data_dir
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "data_science_project")
os.environ["NHL_DATA_DIR"] = str(ROOT / "ift6758" / "data" / "data" / "raw")

In [ ]:
# IL MANQUE: PLAYER AND GOALIE NAMES, POWERPLAY:
# https://api-web.nhle.com/v1/gamecenter/2017020001/play-by-play
# 
# RESTE À NOMMER DONNÉES MANQUANTES ET EXCLURE AU BESOIN
def game_to_pandas(game:dict, season:int):
    """Extraire la liste d'événements d'une seule partie en dataframe pandas"""

    # Liste d'événements
    plays = game["plays"]

    # Conversion en dataframe
    events = pd.json_normalize(plays)

    # Garder chaque tir et chaque but
    keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
    shots = events.loc[keep].copy()

    # Game ID
    shots["game_id"] = game["id"]

    # Game type
    shots["game_type"] = game["gameType"]

    # Nommer les colonnes du dataframe
    column_names = {
        "game_id": "game_id",
        "eventId": "event_id",
        "game_type": "game_type",
        "periodDescriptor.number": "period",
        "periodDescriptor.periodType": "period_type",
        "timeInPeriod": "time_in_period",
        "situationCode": "situation",
        "typeDescKey": "event_type",
        "details.eventOwnerTeamId": "team_id",
        "details.xCoord": "x",
        "details.yCoord": "y",
        "details.shotType": "shot_type",
        "details.shootingPlayerId": "shooter_id",
        "details.scoringPlayerId": "scorer_id",
        "details.goalieInNetId": "goalie_id"
    }

    # Renommer les colonnes
    shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)

    # Noms des équipes
    team_names = {
        game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
        game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
    }
    shots["team"] = shots["team_id"].map(team_names)

    # Noms des joueurs
    """player_names = {}
    for i in range(40):
        first_name = shots["rosterSpots"][{i}]["firstName"]["default"]
        last_name = shots["rosterSpots"][{i}]["lastName"]["default"]
        name = f"{first_name} {last_name}"
        player_names["player_id"] = name
    if shots["shooter_id"]:
        shots["player_name"] = name
    else:
        shots[""]"""

    # Tir/but en supériorité numérique, infériorité numérique ou jeu normal
    """situationCode: 4 digits [gardien visiteur][patineurs visiteurs][patineurs locaux][gardien local]
    premier et dernier digit 0 ou 1 (absent ou présent) // digits du milieu: jusqu'à 6 si gardien au banc"""

    # Indicateur de but ou non
    shots["is_goal"] = shots["event_type"].eq("goal")

    # Indicateur de filet désert ou non
    shots["empty_net"] = shots["goalie_id"].isna()

    # Convertir en valeurs manquantes
    shots = shots.convert_dtypes()
    return shots

def season_to_pandas(season: int):
    """Extraire le dataframe pandas de tous les matchs d'une saison"""
    whole_season = load_season(season, "regular") + load_season(season, "playoffs")

    # Liste de dataframes
    dataframes = []

    # Extraire chaque dataframe d'un match
    for game in whole_season:
        # Claude.AI utilisé, cf annexe
        try:
            df = game_to_pandas(game, season)
        except Exception as e:
            print(f"Skipping game {game}: {e}")
            continue
        
        dataframes.append(df)

    # Soulever une erreur si aucun match n'est extrait
    if not dataframes:
        raise ValueError("Aucune partie trouvée.")

    # Retourner le dataframe avec tous les matchs d'une saison
    return pd.concat(dataframes, ignore_index=True)

    # Une fois la fonction complète, sauvegarder en CVS en remplaçant les chemins de la démo pour les bons:
    """Save the cleaned table to a CSV file.
    processed_dir = Path("data/processed")
    processed_dir.mkdir(parents=True, exist_ok=True)
    csv_path = processed_dir / f"{game_id}_shots.csv"
    shots.to_csv(csv_path, index=False)
    print("Saved processed table:", csv_path.resolve())"""

In [3]:
game_to_pandas(get_game("2017020001"), 2017)

,game_id,event_id,game_type,period,period_type,time_in_period,situation,event_type,team_id,x,y,shot_type,shooter_id,scorer_id,goalie_id,team,is_goal,empty_net
3,2017020001,54,2,1,REG,00:38,1551,shot-on-goal,52,-36,-28,wrist,8477504,<NA>,8475883,WPG,False,False
5,2017020001,55,2,1,REG,00:49,1551,shot-on-goal,52,-75,1,wrist,8473574,<NA>,8475883,WPG,False,False
9,2017020001,58,2,1,REG,01:03,1551,shot-on-goal,52,-73,10,backhand,8473412,<NA>,8475883,WPG,False,False
14,2017020001,62,2,1,REG,01:46,1551,shot-on-goal,10,80,-3,wrist,8470611,<NA>,8473461,TOR,False,False
22,2017020001,68,2,1,REG,03:42,1451,shot-on-goal,52,-44,-21,snap,8479339,<NA>,8475883,WPG,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,2017020001,636,2,3,REG,11:57,1551,goal,10,84,-3,tip-in,<NA>,8479318,8476945,TOR,True,False
283,2017020001,638,2,3,REG,12:57,1551,goal,52,-80,1,wrist,<NA>,8473618,8475883,WPG,True,False
288,2017020001,641,2,3,REG,13:55,1551,shot-on-goal,52,-66,8,wrist,8476460,<NA>,8475883,WPG,False,False
292,2017020001,644,2,3,REG,15:52,1551,shot-on-goal,10,83,-28,backhand,8466139,<NA>,8476945,TOR,False,False


In [4]:
df_2019 = season_to_pandas(2019)

In [ ]:
display(df_2019.isna().sum().rename("missing_count").to_frame())

,missing_count
game_id,0
event_id,0
game_type,0
period,0
period_type,0
time_in_period,0
situation,0
event_type,0
team_id,0
x,1


In [27]:
shots_with_coordinates = df_2019.dropna(subset=["x", "y"])
print("Rows excluded from a coordinate-based analysis:", len(df_2019) - len(shots_with_coordinates))
print("Rows kept in the full shots table:", len(df_2019))

Rows excluded from a coordinate-based analysis: 1
Rows kept in the full shots table: 73867
